# Science images analysis

In [ ]:
import numpy as np
import aotools
import matplotlib.pyplot as plt
import astropy.io.fits as pfits
from IPython.display import HTML
from matplotlib import animation
from scipy.signal import peak_widths

In [ ]:
imgs_ol = pfits.getdata("OL_images.fits").T
imgs_cl = pfits.getdata("CL_images.fits").T
imgs_cl_bad = pfits.getdata("CL_bad_images.fits").T
print(imgs_cl_bad.shape)

In our image files, the first two dimensions are plane coordinates, the last one is time.

## 1. Short exposure images

In [ ]:
plt.imshow(imgs_ol[:, :, 400])
plt.colorbar()

#### A small note on image displays
If we look at the array shape, the firt dimension is the smallest one (240). However on the imshow display, this corresponds to the vertical axis. In addition, the orientation of this axis is top-down, which feel unnatural for plane coordinates.
When looking at images like that, we we use a small homemade function to retrieve a more natural orientation with a transposition (invert x and y axis) and the keyword `origin='lower'` to get the y axis oriented upwards.

In [ ]:
def imdisplay(img, ax=None, **kwargs):
    if ax is not None:
        return ax.imshow(img.T, origin='lower', **kwargs)
    else:
        return plt.imshow(img.T, origin='lower', **kwargs)

imdisplay(imgs_ol[:, :, 400])

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 4))
img1 = imdisplay(imgs_ol[50:178, 100:228, 0].T, ax1)
img2 = imdisplay(imgs_cl[50:178, 100:228, 0].T, ax2)
img3 = imdisplay(imgs_cl_bad[50:178, 100:228, 0].T, ax3)

def animate(i):
    img1.set_data(imgs_ol[50:178, 100:228, i].T)
    img2.set_data(imgs_cl[50:178, 100:228, i].T)
    img3.set_data(imgs_cl_bad[50:178, 100:228, i].T)
    return(img1,img2,)

anim = animation.FuncAnimation(fig, animate,
                               frames=300, interval=25)

In [ ]:
HTML(anim.to_jshtml())

## 2. Long exposure images
In some applications (for example astronomy), the science camera needs to integrate for a long time (10s of seconds, minutes) to take one image. In this case, the relevant information is given by the sum (or average) of many successive short exposure images.
Here it is obtained by averaging along the third axis of the data cube.

In [ ]:
img_le_ol = np.mean(imgs_ol[:, :, 30:], axis=2)
img_le_cl = np.mean(imgs_cl[:, :, 30:], axis=2)
img_le_cl_bad = np.mean(imgs_cl_bad[:, :, 30:], axis=2)

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1); imdisplay(img_le_ol)
plt.subplot(1, 3, 2); imdisplay(img_le_cl)
plt.subplot(1, 3, 3); imdisplay(img_le_cl_bad)

In [ ]:
# Approximate crop to (128, 128) shape
img_le_ol_c = img_le_ol[50:178, 100:228]
img_le_cl_c = img_le_cl[50:178, 100:228]
img_le_cl_bad_c = img_le_cl_bad[50:178, 100:228]

In [ ]:
imdisplay(img_le_ol_c)

In [ ]:
imdisplay(img_le_cl_c[20:-20, 20:-20])

In [ ]:
imdisplay(img_le_cl_bad_c[20:-20, 20:-20])

## 3. Image interpolation

One metric that could be used to assess performance is to look at crosscuts of the psf along horizontal and vertical axis. To do so we can for example find the coordinates of the maximum in the corrected PSF and use it to cross-cut the image.

In [ ]:
(xmax, ymax) = np.unravel_index(np.argmax(img_le_cl_c), img_le_cl_c.shape)
print(xmax, ymax)

In [ ]:
xcut = img_le_cl_c[xmax, :]
plt.plot(xcut, label=f'crosssection at x = {xmax}')
ycut = img_le_cl_c[:, ymax]
plt.plot(ycut, label=f'crosssection at y = {ymax}')
plt.legend()

One observation we can make is that the close loop PSF is not really well sampled. In many cases, it can be usefull to interpolate the image.
To do so, we will use a function provided by aotools, where we have to choose the new image size. This saize can be arbitrary, but it is generally easier to use an integer interpolation factor.

In [ ]:
factor = 4

In [ ]:
(xsize, ysize) = img_le_ol_c.shape
interpolated_ol = aotools.zoom_rbs(img_le_ol_c, (xsize*factor, ysize*factor))
imdisplay(interpolated_ol)

In [ ]:
(xsize, ysize) = img_le_cl_c.shape
interpolated_cl = aotools.zoom_rbs(img_le_cl_c, (xsize*factor, ysize*factor))
imdisplay(interpolated_cl)

In [ ]:
(xsize, ysize) = img_le_cl_bad_c.shape
interpolated_cl_bad = aotools.zoom_rbs(img_le_cl_bad_c, (xsize*factor, ysize*factor))
imdisplay(interpolated_cl_bad)

The image has now more pixels, and it will be easier to look at by cropping it around the maximum. We can define the half width of a window that will be used to crop images and plots around the max value.

In [ ]:
hw = 23 #semi-size of the image
(xmax_i, ymax_i) = np.unravel_index(np.argmax(interpolated_cl), interpolated_cl.shape) #coords of max in interpolated image
print(xmax_i, ymax_i)
plt.subplot(1, 2, 1)
imdisplay(interpolated_ol[xmax_i-hw:xmax_i+hw, ymax_i-hw:ymax_i+hw])
plt.subplot(1, 2, 2)
imdisplay(interpolated_cl[xmax_i-hw:xmax_i+hw, ymax_i-hw:ymax_i+hw])

## 3. Axis units

As mentioned in the simulation workshop, the axis units for camera images are pixels. However this is strongly dependent on the full optical setup and might not be the relevant information. For example in astronomy where most of the objects are considered at infinity, a the interesting spatial unit is an angle (expressed in arcsecond, arcminutes or in units of $\lambda/D$).

This will change for every system.

In our case, the image is formed in the focal plane of a lense of focal length 180 mm, so the pixel size in radians is approximately the pixel size in mm divided by the focal length in mm.

On interpolated images, the pixel unit is divided by the interpolation ratio.

In [ ]:
pxsize = (7.4e-6/180e-3)*(180/np.pi)*3600 # in arcseconds
pxsize = (7.4e-6/180e-3)*(12e-3/635e-9) # in lambda/D unit

In [ ]:
hw = 30 #semi-size of the image
axis = (np.arange(2*hw)-hw)*pxsize/factor #interpolated image
imdisplay(interpolated_cl[xmax_i-hw:xmax_i+hw, ymax_i-hw:ymax_i+hw], 
           extent=[axis[0], axis[-1], axis[0], axis[-1]])
plt.colorbar()

## 4. The noise problem

Contrary to what we see in simulation, in the real camera images there is usually no pixel that reach a real null value.
To perform analysis such as center of gravity or encircled energy, the noise floor needs to be removed.

In [ ]:
imdisplay(interpolated_cl[:180, :])
plt.colorbar()
np.max(interpolated_cl[:180, :])

In [ ]:
denoised_cl = interpolated_cl - 0.0018
denoised_cl[denoised_cl < 0] = 0
imdisplay(np.log10(denoised_cl))
plt.colorbar()

In [ ]:
denoised_ol = interpolated_ol - 0.0016
denoised_ol[denoised_ol < 0] = 0
imdisplay(np.log10(denoised_ol))
plt.colorbar()

In [ ]:
denoised_cl_bad = interpolated_cl_bad - 0.02 # The noise is much higher in this dataset
denoised_cl_bad[denoised_cl_bad < 0] = 0
imdisplay(np.log10(denoised_cl_bad))
plt.colorbar()

## 5. Cross section, full width at half maximum

Now that we have good resolution images, and a meaningful unit for the pixes size, we can compare the performance by looking at the cross section for our 3 cases.

In [ ]:
hw = 200 #semi-size of the image
axis = (np.arange(2*hw)-hw)*pxsize/factor
xcut = denoised_ol[xmax_i, :]
plt.plot(axis, xcut[ymax_i-hw: ymax_i+hw], label=f'crosssection at x = {xmax_i}')
ycut = denoised_ol[:, ymax_i]
plt.plot(axis, ycut[xmax_i-hw: xmax_i+hw], label=f'crosssection at y = {ymax_i}')
plt.legend(); plt.grid()

In [ ]:
hw = 32 #semi-size of the image
axis = (np.arange(2*hw)-hw)*pxsize/factor
xcut = denoised_cl[xmax_i, :]
plt.plot(axis, xcut[ymax_i-hw: ymax_i+hw], label=f'crosssection at x = {xmax_i}')
ycut = denoised_cl[:, ymax_i]
plt.plot(axis, ycut[xmax_i-hw: xmax_i+hw], label=f'crosssection at y = {ymax_i}')
plt.legend(); plt.grid()
(wx, hx, lx, rx) = peak_widths(xcut, [ymax_i])
(wy, hy, ly, ry) = peak_widths(ycut, [xmax_i])
print(f"x FWHM = {pxsize/factor * wx[0]}, y FWHM = {pxsize/factor * wy[0]}")

In [ ]:
hw = 100 #semi-size of the image
axis = (np.arange(2*hw)-hw)*pxsize/factor
xcut = denoised_cl_bad[xmax_i, :]
plt.plot(axis, xcut[ymax_i-hw: ymax_i+hw], label=f'crosssection at x = {xmax_i}')
ycut = denoised_cl_bad[:, ymax_i]
plt.plot(axis, ycut[xmax_i-hw: xmax_i+hw], label=f'crosssection at y = {ymax_i}')
plt.legend(); plt.grid()